In [ ]:
!pip install SPARQLWrapper

In [ ]:
import time
import json
from SPARQLWrapper import SPARQLWrapper, JSON
import re

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"

def run_query(query: str) -> list[dict]:
    sparql = SPARQLWrapper(WIKIDATA_ENDPOINT)
    sparql.addCustomHttpHeader("User-Agent", "EuropeOntologyCreationBot/1.0 (research)")
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    time.sleep(1)
    return sparql.query().convert()["results"]["bindings"]


In [ ]:
CITIES_QUERY = """
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd:  <http://www.wikidata.org/entity/>
SELECT ?city ?cityLabel ?countryLabel (SAMPLE(?population) AS ?population) (SAMPLE(?coord) AS ?coord)
WHERE {
  ?city wdt:P31/wdt:P279* wd:Q515 ;
        wdt:P17 ?country ;
        wdt:P1082 ?population .
  ?country wdt:P30 wd:Q46 .
  FILTER(?population >= 500000 || EXISTS { ?country wdt:P36 ?city })
  MINUS { ?city wdt:P31 wd:Q3024240 }
  MINUS { ?city wdt:P17 / wdt:P31 wd:Q3024240 }
  OPTIONAL { ?city wdt:P625 ?coord }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en, [AUTO_LANGUAGE], mul" }
}
GROUP BY ?city ?cityLabel ?countryLabel
ORDER BY DESC(?population)
"""
def places_query(city_qid: str) -> str:
    return f"""
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd:  <http://www.wikidata.org/entity/>
SELECT DISTINCT ?place ?placeLabel ?typeLabel ?coord ?inception
WHERE {{
  ?place wdt:P131* wd:{city_qid} .
  ?place wdt:P31/wdt:P279* ?type .
  VALUES ?type {{
    wd:Q570116 wd:Q839954 wd:Q4989906 wd:Q33506
    wd:Q16560  wd:Q44613  wd:Q2977    wd:Q23413
    wd:Q12518  wd:Q174782 wd:Q24354   wd:Q22698
    wd:Q23397  wd:Q4022   wd:Q483110  wd:Q118554787
    wd:Q40080  wd:Q34038  wd:Q8502    wd:Q132510
    wd:Q16970  wd:Q34627  wd:Q32815
  }}
  OPTIONAL {{ ?place wdt:P625 ?coord }}
  OPTIONAL {{ ?place wdt:P571 ?inception }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en,[AUTO_LANGUAGE], mul" }}
}}
"""

def build_ontology():
    print("Fetching European cities...")
    cities_raw = run_query(CITIES_QUERY)

    ontology = {}
    wrong_instances = ["Madrid city", "city of belgrade", "Metropolis of Lyon",
                       "Brussels-Capital Region", "Dublin city"]
    rename_map = {
        "Bern": "Berne"
    }

    for c in cities_raw:
        qid   = c["city"]["value"].split("/")[-1]
        label = c.get("cityLabel", {}).get("value", qid)

        if label in wrong_instances:
          continue

        if label in rename_map.keys():
          label = rename_map[label]

        if label.lower().startswith("city of"):
          label = label[8:].strip()

        if label.lower().endswith("city"):
          label = label[:-4].strip()

        if label.lower().endswith("urban area"):
          label = label[:-10].strip()

        if re.search("^Q[0-9]*", label):
          print(label)
          continue # remove entity without label

        ontology[label] = {
            "QID": qid,
            "country":    c.get("countryLabel", {}).get("value"),
            "population": c.get("population",   {}).get("value"),
            "coord":      c.get("coord",        {}).get("value"),
            "places":     []
        }

    print(f"Found {len(ontology)} cities. Fetching POIs...")

    for i, (label, data) in enumerate(ontology.items()):
        print(f"  [{i+1}/{len(ontology)}] {label}")
        try:
            places_raw = run_query(places_query(data["QID"]))
            data["places"] = []
            for p in places_raw:
              if re.search("^Q[0-9]*", p.get("placeLabel",  {}).get("value")):
                continue # remove entity without label
              typeL = p.get("typeLabel",   {}).get("value")

              if typeL.lower().endswith("building"):
                  typeL = typeL[:-8].strip()

              data["places"].append(
                  {
                      "label":     p.get("placeLabel",  {}).get("value"),
                      "type":      typeL, # format correcty
                      "coord":     p.get("coord",       {}).get("value"),
                      "inception": p.get("inception",   {}).get("value"),
                  })
        except Exception as e:
            print(f" Error for {label}: {e}")
        time.sleep(2)

    return ontology

if __name__ == "__main__":
    ontology = build_ontology()
    with open("europe_ontology.json", "w", encoding="utf-8") as f:
        json.dump(ontology, f, ensure_ascii=False, indent=2)
    print("Saved to europe_ontology.json")
